In [1]:
import numpy as np
import pandas as pd
import re

In [ ]:
filename_template = R'template.csv'
filename_data = R'input.xlsx'

In [3]:
template = pd.read_csv(filename_template, on_bad_lines='skip')
data = pd.read_excel(filename_data, skiprows=1)

In [4]:
def get_units(df):
    units = {}
    for column in df.columns:
        match = re.match(r'^(.*\S) ?\(([^()]+)\)$', column)
        if match:
            [col, unit] = match.groups()
            units[col] = unit
    return units

units = get_units(template)
units_origin = get_units(data)

In [5]:
def empties(df, cols):
    for col in cols:
        df[col] = ''
    return df

def matches(df, cols):
    for col in cols:
        origins = cols[col]
        unit = units[col]
        if not origins:
            df[f'{col} ({unit})'] = ''
            continue
        if not isinstance(origins, list):
            origins = [origins]
        sum = 0
        for origin in origins:
            unit_origin = units_origin[origin]
            if unit == unit_origin:
                times = 1
            elif unit == f"m{unit_origin}":
                times = 0.001
            elif f"m{unit}" == unit_origin:
                times = 1000
            sum += df[f'{origin}({unit_origin})'] * times
        df[f'{col} ({unit})'] = sum
    return df

def slashes(col):
    return col.str.replace(',', '/')

def replaces(col):
    return col.str.replace(',', '，').str.replace(':', '：').str.replace(';', '；')

In [6]:
units_origin

{'廢棄率': '%',
 '熱量': 'kcal',
 '修正熱量': 'kcal',
 '水分': 'g',
 '粗蛋白': 'g',
 '粗脂肪': 'g',
 '飽和脂肪': 'g',
 '灰分': 'g',
 '總碳水化合物': 'g',
 '膳食纖維': 'g',
 '糖質總量': 'g',
 '葡萄糖': 'g',
 '果糖': 'g',
 '半乳糖': 'g',
 '麥芽糖': 'g',
 '蔗糖': 'g',
 '乳糖': 'g',
 '鈉': 'mg',
 '鉀': 'mg',
 '鈣': 'mg',
 '鎂': 'mg',
 '鐵': 'mg',
 '鋅': 'mg',
 '磷': 'mg',
 '銅': 'mg',
 '錳': 'mg',
 '維生素A總量': 'IU',
 '視網醇當量(RE)': 'ug',
 '視網醇': 'ug',
 'α-胡蘿蔔素': 'ug',
 'β-胡蘿蔔素': 'ug',
 '維生素D總量': 'ug',
 '維生素D2': 'ug',
 '維生素D3': 'ug',
 '維生素E總量': 'mg',
 'α-維生素E當量(α-TE)': 'mg',
 'α-生育酚': 'mg',
 'β-生育酚': 'mg',
 'γ-生育酚': 'mg',
 'δ-生育酚': 'mg',
 '維生素K1': 'ug',
 '維生素K2 (MK-4)': 'ug',
 '維生素K2 (MK-7)': 'ug',
 '維生素B1': 'mg',
 '維生素B2': 'mg',
 '菸鹼素': 'mg',
 '維生素B6': 'mg',
 '維生素B12': 'ug',
 '葉酸': 'ug',
 '維生素C': 'mg',
 '脂肪酸S總量': 'mg',
 '酪酸(4:0)': 'mg',
 '己酸(6:0)': 'mg',
 '辛酸(8:0)': 'mg',
 '癸酸(10:0)': 'mg',
 '月桂酸(12:0)': 'mg',
 '十三酸(13:0)': 'mg',
 '肉豆蔻酸(14:0)': 'mg',
 '十五酸(15:0)': 'mg',
 '棕櫚酸(16:0)': 'mg',
 '十七酸(17:0)': 'mg',
 '硬脂酸(18:0)': 'mg',
 '十九酸(19:0)': 'mg',
 '花生

In [7]:
data['俗名'] = slashes(data['俗名'].fillna(''))
data['Name'] = data.apply(lambda row: row['樣品名稱'] if row['俗名'] == '' else f"{row['樣品名稱']}({row['俗名']})", axis=1)
data['Note'] = data['食品分類'] + "；" + replaces(data['內容物描述'])
data['Is Liquid'] = 0
data['Source URL'] = 'https://consumer.fda.gov.tw/Food/TFND.aspx?nodeID=178 @2025.07.10'
data = empties(data, [
    'Brand', 
    'Barcode'
])
data = matches(data, {
    'Proteins': '粗蛋白',
    'Carbohydrates': '總碳水化合物',
    'Package Weight': None, 
    'Serving Weight': None, 
    'Energy': '熱量',
    'Fats': '粗脂肪',
    'Saturated Fats': '飽和脂肪',
    'Trans Fats': '反式脂肪',
    'Monounsaturated Fats': '脂肪酸M總量',
    'Polyunsaturated Fats': '脂肪酸P總量',
    'Omega-3': ['次亞麻油酸(18:3)', '廿碳五烯酸(20:5)', '廿二碳五烯酸(22:5)', '廿二碳六烯酸(22:6)'],
    'Omega-6': ['亞麻油酸(18:2)', '花生油酸(20:4)'],
    'Sugars': '糖質總量',
    'Added Sugars': None,
    'Dietary Fiber': '膳食纖維',
    'Soluble Fiber': None,
    'Insoluble Fiber': None,
    'Salt': None,
    'Cholesterol': '膽固醇',
    'Caffeine': None,
    'Vitamin A': '維生素A總量',
    'Vitamin B1': '維生素B1',
    'Vitamin B2': '維生素B2',
    'Vitamin B3': '菸鹼素',
    'Vitamin B5': None,
    'Vitamin B6': '維生素B6',
    'Vitamin B7': None,
    'Vitamin B9': '葉酸',
    'Vitamin B12': '維生素B12',
    'Vitamin C': '維生素C',
    'Vitamin D': '維生素D總量',
    'Vitamin E': '維生素E總量',
    'Vitamin K': '維生素K1',
    'Manganese': '錳',
    'Magnesium': '鎂',
    'Potassium': '鉀',
    'Calcium': '鈣',
    'Copper': '銅',
    'Zinc': '鋅',
    'Sodium': '鈉',
    'Iron': '鐵',
    'Phosphorus': '磷',
    'Selenium': None,
    'Iodine': None,
    'Chromium': None
})

C:\Users\Travis\AppData\Local\Temp\ipykernel_3440\3699915245.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Name'] = data.apply(lambda row: row['樣品名稱'] if row['俗名'] == '' else f"{row['樣品名稱']}({row['俗名']})", axis=1)
C:\Users\Travis\AppData\Local\Temp\ipykernel_3440\3699915245.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Note'] = data['食品分類'] + "；" + replaces(data['內容物描述'])
C:\Users\Travis\AppData\Local\Temp\ipykernel_3440\3699915245.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually

In [8]:
data[template.columns].to_csv(R'output.foodyou.csv', encoding='utf-8', index=False)